# Clase 215 — Star schema completo con DuckDB

Construimos un star schema de e-commerce: `dim_date`, `dim_customer` (SCD 2), `dim_product`, `dim_store`, `fact_sales`. Demostramos SCD 2 con cliente que cambia de ciudad.

In [ ]:
import duckdb, tempfile
from pathlib import Path
DB = str(Path(tempfile.gettempdir()) / 'dw_star.duckdb')
Path(DB).unlink(missing_ok=True)
con = duckdb.connect(DB)

## 1. `dim_date` — 5 años de fechas precalculadas

In [ ]:
con.execute('''
    CREATE TABLE dim_date AS
    SELECT
        CAST(strftime(d, '%Y%m%d') AS INT) AS date_key,
        d                                  AS date,
        EXTRACT('year'  FROM d)            AS year,
        EXTRACT('quarter' FROM d)          AS quarter,
        EXTRACT('month' FROM d)            AS month,
        strftime(d, '%B')                  AS month_name,
        EXTRACT('day'   FROM d)            AS day,
        EXTRACT('isodow' FROM d)           AS day_of_week_iso,
        strftime(d, '%A')                  AS day_name,
        CASE WHEN EXTRACT('isodow' FROM d) IN (6, 7) THEN TRUE ELSE FALSE END AS is_weekend,
        CASE WHEN EXTRACT('month' FROM d) >= 10 THEN EXTRACT('year' FROM d) + 1
             ELSE EXTRACT('year' FROM d) END AS fiscal_year
    FROM (SELECT UNNEST(generate_series(DATE '2022-01-01', DATE '2026-12-31', INTERVAL 1 DAY)) AS d)
''')
print(con.execute('SELECT COUNT(*), MIN(date), MAX(date) FROM dim_date').fetchone())
print(con.execute("SELECT * FROM dim_date WHERE date='2026-06-17'").fetchdf())

## 2. `dim_product` — denormalizada

In [ ]:
con.execute('''
    CREATE TABLE dim_product AS
    SELECT
        ROW_NUMBER() OVER () AS product_key,         -- surrogate key
        sku,                                          -- natural key
        name, category, brand, list_price
    FROM (VALUES
        ('SKU-001', 'Coffee Mug',   'Kitchen',  'AcmeBrand',  12.0),
        ('SKU-002', 'T-Shirt',      'Apparel',  'BlueLabel',  25.0),
        ('SKU-003', 'Notebook',     'Office',   'NoteCo',      8.0),
        ('SKU-004', 'Headphones',   'Electronics', 'AudioPro', 150.0),
        ('SKU-005', 'Water Bottle', 'Kitchen',  'AcmeBrand',  18.0)
    ) v(sku, name, category, brand, list_price)
''')
print(con.execute('SELECT * FROM dim_product').fetchdf())

## 3. `dim_customer` — SCD Tipo 2

In [ ]:
con.execute('''
    CREATE TABLE dim_customer (
        customer_key INT PRIMARY KEY,
        customer_id  TEXT,            -- natural key
        name TEXT, email TEXT,
        city TEXT, country TEXT,
        valid_from DATE NOT NULL,
        valid_to   DATE,              -- NULL = vigente
        is_current BOOLEAN NOT NULL
    )
''')

# Carga inicial (estado al 2024-01-01)
con.execute("""
    INSERT INTO dim_customer VALUES
        (1, 'C001', 'Alice', 'a@x.com', 'Buenos Aires', 'AR', DATE '2024-01-01', NULL, TRUE),
        (2, 'C002', 'Bob',   'b@x.com', 'Montevideo',   'UY', DATE '2024-01-01', NULL, TRUE),
        (3, 'C003', 'Carol', 'c@x.com', 'Santiago',     'CL', DATE '2024-01-01', NULL, TRUE)
""")
print(con.execute('SELECT * FROM dim_customer').fetchdf())

## 4. `fact_sales` — grain: 1 row per order line

In [ ]:
con.execute('''
    CREATE TABLE fact_sales (
        sale_id      BIGINT,
        date_key     INT REFERENCES dim_date,
        product_key  INT REFERENCES dim_product,
        customer_key INT REFERENCES dim_customer,
        qty INT, revenue DOUBLE, discount DOUBLE
    )
''')

# 100 ventas sintéticas
con.execute("""
    INSERT INTO fact_sales
    SELECT
        i AS sale_id,
        CAST(strftime(DATE '2024-06-01' + INTERVAL ((random()*100)::INT) DAY, '%Y%m%d') AS INT) AS date_key,
        (1 + (random() * 4)::INT) AS product_key,
        (1 + (random() * 2)::INT) AS customer_key,
        (1 + (random() * 5)::INT) AS qty,
        20 + random() * 100       AS revenue,
        random() * 5              AS discount
    FROM range(100) t(i)
""")
print(con.execute('SELECT COUNT(*), SUM(revenue) FROM fact_sales').fetchone())

## 5. SCD 2 en acción: Alice se muda Buenos Aires → Madrid

In [ ]:
# Pre-move: ventas de Alice se asocian a Buenos Aires
pre = con.execute("""
    SELECT c.city, COUNT(*) sales, SUM(f.revenue) rev
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    WHERE c.customer_id = 'C001'
    GROUP BY c.city
""").fetchdf()
print('Antes del cambio:')
print(pre)

# SCD 2 update: cerrar fila vigente, insertar nueva
con.execute("""
    UPDATE dim_customer
    SET valid_to = DATE '2024-09-01', is_current = FALSE
    WHERE customer_id = 'C001' AND is_current = TRUE
""")
con.execute("""
    INSERT INTO dim_customer VALUES (4, 'C001', 'Alice', 'a@x.com', 'Madrid', 'ES', DATE '2024-09-02', NULL, TRUE)
""")

# Nuevas ventas posteriores referencian customer_key=4 (Madrid)
con.execute("""
    INSERT INTO fact_sales VALUES
        (1001, 20240910, 1, 4, 2, 50.0, 0),
        (1002, 20240915, 2, 4, 1, 25.0, 0)
""")

post = con.execute("""
    SELECT c.city, COUNT(*) sales, SUM(f.revenue) rev
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    WHERE c.customer_id = 'C001'
    GROUP BY c.city
    ORDER BY rev DESC
""").fetchdf()
print('\nDespués del cambio (ventas correctamente atribuidas a la ciudad de cada momento):')
print(post)

## 6. Query analítica clásica usando todas las dims

In [ ]:
result = con.execute('''
    SELECT
        p.brand,
        d.fiscal_year,
        d.is_weekend,
        COUNT(*)            AS n_orders,
        SUM(f.revenue)      AS revenue,
        AVG(f.revenue)      AS avg_order
    FROM fact_sales f
    JOIN dim_product  p USING (product_key)
    JOIN dim_date     d USING (date_key)
    GROUP BY p.brand, d.fiscal_year, d.is_weekend
    ORDER BY revenue DESC
    LIMIT 10
''').fetchdf()
print(result)

In [ ]:
con.close(); print('done.')

## Ejercicio guiado

1. Definí el grain de tu fact table para un caso real propio. Justificá.
2. Agregá una `dim_store` con jerarquía geográfica (store → city → country). Decidí star (denormalizada) o snowflake (separar `dim_city` y `dim_country`).
3. Implementá la query: cohort retention mensual usando `dim_date` + `fact_sales`.
4. Convertí el script a dbt models: `models/staging/`, `models/marts/dim_*.sql`, `models/marts/fact_*.sql`. Agregá `tests` (unique, not_null, accepted_values).
5. Bonus: mismo schema en BigQuery con `PARTITION BY date_key` + `CLUSTER BY product_key, customer_key`. Compará costo de queries.

## Conclusiones

- Star schema sigue siendo el patrón ganador en 2026 para data warehouses.
- Grain definido al inicio = decisión que evita los bugs más caros.
- SCD 2 con surrogate keys = único modo correcto de manejar history de entidades.
- `dim_date` precalculada > derivar en query.
- **Fin de Parte 5**: tenés orquestación (208-209) + procesamiento (210-211) + DW (212) + streaming (213) + formatos (214) + modelado (215) = stack completo para alimentar ML a escala.

## ✅ Soluciones de los ejercicios

Todo el modelado dimensional se construye con **DuckDB** (instalado): definimos el *grain*,
generamos una `dim_date`, montamos un *star schema* completo, aplicamos **SCD Tipo 2** y
corremos la query analítica típica. Datos sintéticos de e-commerce, sin internet.

### Ejercicio 1 — Identificar el grain

El *grain* es "qué representa una fila del fact". Para pedidos con varias líneas, el grain
más útil suele ser **1 fila por línea de pedido** (permite analizar por producto). Lo
demostramos: si el grain fuera "1 fila por pedido" perderíamos el detalle por producto.

In [ ]:
import duckdb
con = duckdb.connect()

con.execute("""
CREATE TABLE raw_order_lines AS
SELECT * FROM (VALUES
    (1001, 'A', 2, 10.0),
    (1001, 'B', 1, 25.0),
    (1002, 'A', 3, 10.0)
) AS t(order_id, product, qty, price)
""")

grain_line  = con.execute("SELECT COUNT(*) FROM raw_order_lines").fetchone()[0]
grain_order = con.execute("SELECT COUNT(DISTINCT order_id) FROM raw_order_lines").fetchone()[0]
print(f"grain '1 fila por linea' = {grain_line} filas | grain '1 fila por pedido' = {grain_order} filas")

# con grain por-línea conservamos el detalle producto; con grain por-pedido lo perdemos
assert grain_line == 3 and grain_order == 2
print("Elegimos grain = 1 fila por LINEA de pedido (maximo detalle analizable)")
print("OK ejercicio 1 — grain definido y justificado")

### Ejercicio 2 — `dim_date`

Una dimensión de fecha con 5 años de días y atributos derivados (`year`, `quarter`, `month`,
`day_of_week_iso`, `is_weekend`, `is_holiday_us`, `fiscal_year`). Usamos `range()` de DuckDB.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_date AS
SELECT
    CAST(strftime(d, '%Y%m%d') AS INTEGER)      AS date_key,
    d                                           AS full_date,
    year(d)                                     AS year,
    quarter(d)                                  AS quarter,
    month(d)                                    AS month,
    isodow(d)                                   AS day_of_week_iso,
    isodow(d) IN (6, 7)                         AS is_weekend,
    (month(d) = 1  AND day(d) = 1) OR
    (month(d) = 7  AND day(d) = 4) OR
    (month(d) = 12 AND day(d) = 25)             AS is_holiday_us,
    CASE WHEN month(d) >= 10 THEN year(d) + 1 ELSE year(d) END AS fiscal_year
FROM range(DATE '2020-01-01', DATE '2025-01-01', INTERVAL 1 DAY) AS t(d)
""")

n_days = con.execute("SELECT COUNT(*) FROM dim_date").fetchone()[0]
n_weekend = con.execute("SELECT COUNT(*) FROM dim_date WHERE is_weekend").fetchone()[0]
print("dias en dim_date:", n_days, "| fines de semana:", n_weekend)
sample = con.execute("SELECT full_date, day_of_week_iso, is_weekend, is_holiday_us FROM dim_date "
                     "WHERE full_date IN (DATE '2024-07-04', DATE '2024-07-06')").df()
print(sample)

assert n_days == 1827, "5 años (2020-2024) incluye 2 bisiestos = 1827 días"
assert con.execute("SELECT is_holiday_us FROM dim_date WHERE full_date=DATE '2024-07-04'").fetchone()[0]
print("OK ejercicio 2 — dim_date con atributos de calendario generada")

### Ejercicio 3 — Star schema

`fact_sales` en el centro, rodeado de `dim_product`, `dim_customer`, `dim_store`, `dim_date`.
El fact guarda **foreign keys + métricas**; las dims guardan los atributos descriptivos.

In [ ]:
con.execute("CREATE TABLE dim_product (product_key INT, sku VARCHAR, brand VARCHAR, category VARCHAR)")
con.execute("CREATE TABLE dim_customer (customer_key INT, name VARCHAR, city VARCHAR)")
con.execute("CREATE TABLE dim_store (store_key INT, store_name VARCHAR, region VARCHAR)")
con.execute("INSERT INTO dim_product VALUES "
            "(1,'SKU-A','Acme','Tools'),(2,'SKU-B','Globex','Home'),(3,'SKU-C','Acme','Home')")
con.execute("INSERT INTO dim_customer VALUES (1,'Ana','Lima'),(2,'Beto','Quito')")
con.execute("INSERT INTO dim_store VALUES (1,'Centro','Andes'),(2,'Norte','Costa')")

con.execute("""
CREATE TABLE fact_sales (
    date_key INT, product_key INT, customer_key INT, store_key INT,
    qty INT, revenue DOUBLE, discount DOUBLE
)""")
con.execute("""INSERT INTO fact_sales VALUES
    (20240705,1,1,1, 2, 20.0, 1.0),
    (20240705,3,1,2, 1, 30.0, 0.0),
    (20240706,2,2,1, 5, 75.0, 5.0),
    (20240708,1,2,2, 3, 30.0, 0.0)""")

joined = con.execute("""
SELECT d.full_date, p.brand, c.name, s.region, f.qty, f.revenue
FROM fact_sales f
JOIN dim_date d     ON f.date_key = d.date_key
JOIN dim_product p  ON f.product_key = p.product_key
JOIN dim_customer c ON f.customer_key = c.customer_key
JOIN dim_store s    ON f.store_key = s.store_key
ORDER BY d.full_date
""").df()
print(joined)

assert len(joined) == 4, "el fact hace join limpio contra las 4 dimensiones"
assert set(joined["brand"]) == {"Acme", "Globex"}
print("OK ejercicio 3 — star schema con fact + 4 dims, joins correctos")

### Ejercicio 4 — SCD Tipo 2 en `dim_customer`

Cuando un cliente cambia de ciudad, **no pisamos** la fila: cerramos la vigente
(`valid_to`, `is_current=FALSE`) e insertamos una nueva (`valid_from`, `is_current=TRUE`).
Así cada venta queda asociada a la ciudad **del momento de la compra**.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_customer_scd (
    surrogate_key INT, customer_id INT, name VARCHAR, city VARCHAR,
    valid_from DATE, valid_to DATE, is_current BOOLEAN
)""")
con.execute("INSERT INTO dim_customer_scd VALUES "
            "(1, 100, 'Ana', 'Lima', DATE '2020-01-01', DATE '9999-12-31', TRUE)")

# Ana se muda a 'Cusco' el 2024-06-01 -> SCD2
change_date = "DATE '2024-06-01'"
con.execute(f"UPDATE dim_customer_scd SET valid_to={change_date}, is_current=FALSE "
            "WHERE customer_id=100 AND is_current")
con.execute(f"INSERT INTO dim_customer_scd VALUES "
            f"(2, 100, 'Ana', 'Cusco', {change_date}, DATE '9999-12-31', TRUE)")

hist = con.execute("SELECT surrogate_key, city, valid_from, valid_to, is_current "
                   "FROM dim_customer_scd WHERE customer_id=100 ORDER BY valid_from").df()
print(hist)

n_current = con.execute("SELECT COUNT(*) FROM dim_customer_scd "
                        "WHERE customer_id=100 AND is_current").fetchone()[0]
assert len(hist) == 2, "dos versiones históricas de Ana"
assert n_current == 1, "exactamente una fila vigente"
city_2023 = con.execute("SELECT city FROM dim_customer_scd WHERE customer_id=100 "
                        "AND DATE '2023-05-01' >= valid_from AND DATE '2023-05-01' < valid_to").fetchone()[0]
assert city_2023 == "Lima"
print("OK ejercicio 4 — SCD Tipo 2: historial preservado, una sola fila vigente")

### Ejercicio 5 — Query analítica típica

"Revenue por brand × month × is_weekend": un solo `SELECT` con joins fact + dims + dim_date.
El modelo dimensional lo hace legible; sin modelar necesitarías más subqueries y casts.

In [ ]:
report = con.execute("""
SELECT p.brand,
       d.month,
       d.is_weekend,
       SUM(f.revenue) AS revenue,
       SUM(f.qty)     AS units
FROM fact_sales f
JOIN dim_date d    ON f.date_key = d.date_key
JOIN dim_product p ON f.product_key = p.product_key
GROUP BY p.brand, d.month, d.is_weekend
ORDER BY revenue DESC
""").df()
print(report)

total = con.execute("SELECT SUM(revenue) FROM fact_sales").fetchone()[0]
assert abs(report["revenue"].sum() - total) < 1e-9, "la agregación cubre todo el revenue"
assert set(report.columns) == {"brand", "month", "is_weekend", "revenue", "units"}
print("OK ejercicio 5 — revenue por brand x month x is_weekend con joins al star schema")